# 01 Data Validation

Initial validation of the expanded raw synthetic ecommerce dataset, including customer, transaction, product, and product-event files.


In [1]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw" / "customers.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
from IPython.display import display

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
    plt.style.use("seaborn-v0_8-whitegrid")
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
TRANSACTIONS_PATH = RAW_DIR / "transactions.csv"
PRODUCTS_PATH = RAW_DIR / "products.csv"
PRODUCT_EVENTS_PATH = RAW_DIR / "product_events.csv"

EXPECTED_START_DATE = pd.Timestamp("2024-01-01")
EXPECTED_END_DATE = pd.Timestamp("2025-12-31")

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.4f}".format)


## Load Raw Data


In [2]:
customers = pd.read_csv(
    CUSTOMERS_PATH,
    parse_dates=["first_purchase_date", "last_transaction_date"],
)
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    parse_dates=["transaction_date"],
)
products = pd.read_csv(PRODUCTS_PATH)
product_events = pd.read_csv(
    PRODUCT_EVENTS_PATH,
    parse_dates=["event_date"],
)


def normalize_boolean(series):
    """Normalize CSV boolean values while preserving invalid values as missing."""
    if pd.api.types.is_bool_dtype(series):
        return series
    return series.astype(str).str.lower().map({"true": True, "false": False})


boolean_columns = {
    "customers": ["price_increase_occurred", "churned"],
    "transactions": ["price_increase_occurred", "churned"],
    "products": ["stockout_flag"],
    "product_events": ["stockout_flag", "price_increase_occurred"],
}
frames = {
    "customers": customers,
    "transactions": transactions,
    "products": products,
    "product_events": product_events,
}

for dataset_name, columns in boolean_columns.items():
    for column in columns:
        frames[dataset_name][column] = normalize_boolean(frames[dataset_name][column])

for dataset_name, frame in frames.items():
    print(f"{dataset_name}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")

display(customers.head())
display(transactions.head())
display(products.head())
display(product_events.head())


customers: 12,500 rows x 13 columns
transactions: 48,769 rows x 22 columns
products: 41 rows x 15 columns
product_events: 29,971 rows x 18 columns


,customer_id,first_purchase_date,customer_region,acquisition_channel,customer_tenure_days,purchase_frequency,prior_spending,transaction_count,total_spending,average_discount_percent,price_increase_occurred,churned,last_transaction_date
0,C000001,2025-10-16,uk,direct,76,8.0000,103.9800,2,103.9800,0.0000,False,True,2025-11-08
1,C000002,2025-07-12,uk,influencer,172,4.2470,120.8300,2,120.8300,12.5000,False,False,2025-10-31
2,C000003,2025-07-12,north_america,influencer,172,6.3710,170.7700,3,170.7700,6.6700,False,False,2025-12-25
3,C000004,2024-08-08,rest_of_world,influencer,510,5.7290,491.9300,8,491.9300,8.7500,False,False,2025-12-13
4,C000005,2025-07-03,uk,affiliate,181,4.0360,104.9700,2,104.9700,0.0000,True,False,2025-07-29


,transaction_id,customer_id,transaction_date,product_id,product_name,product_category,product_price,unit_cost,list_price,selling_price,discount_amount,discount_percent,quantity,order_value,gross_margin,customer_region,acquisition_channel,customer_tenure_days,purchase_frequency,prior_spending,price_increase_occurred,churned
0,T00000001,C000001,2025-10-16,P0034,Core Puffer Vest,outerwear,81.9900,30.4900,81.9900,81.9900,0.0000,0.0000,1,81.9900,51.5000,uk,direct,0,0.0000,0.0000,False,True
1,T00000002,C000001,2025-11-08,P0037,Core Crew Socks,accessories,21.9900,7.7700,21.9900,21.9900,0.0000,0.0000,1,21.9900,14.2200,uk,direct,23,4.0000,81.9900,False,True
2,T00000003,C000002,2025-07-12,P0020,Motion Zip Hoodie,hoodies,66.9900,28.3000,66.9900,56.9400,10.0500,15.0000,1,56.9400,28.6400,uk,influencer,0,0.0000,0.0000,False,False
3,T00000004,C000002,2025-10-31,P0021,Lift Oversized Hoodie,hoodies,70.9900,26.7000,70.9900,63.8900,7.1000,10.0000,1,63.8900,37.1900,uk,influencer,111,3.2910,56.9400,False,False
4,T00000005,C000003,2025-07-12,P0003,Motion Seamless Leggings,leggings,54.9900,20.1900,54.9900,54.9900,0.0000,0.0000,1,54.9900,34.8000,north_america,influencer,0,0.0000,0.0000,False,False


,product_id,product_name,product_category,unit_cost,list_price,gross_margin,gross_margin_rate,total_revenue,total_units_sold,total_purchases,product_views,add_to_cart_events,checkout_started_events,inventory_level,stockout_flag
0,P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,"63,143.2500",0.6266,"100,767.5500",2065,1507,195059,19763,7967,79,True
1,P0002,Core Flex Leggings,leggings,25.2000,53.9900,"38,464.4400",0.4918,"78,204.8400",1577,1214,170860,16926,6741,408,False
2,P0003,Motion Seamless Leggings,leggings,20.1900,54.9900,"58,885.7500",0.5976,"98,538.9100",1964,1464,205376,20769,8377,424,False
3,P0004,Lift Training Leggings,leggings,27.0700,55.9900,"53,752.4300",0.4715,"114,010.2500",2226,1681,208741,20888,8335,89,True
4,P0005,Contour High-Rise Leggings,leggings,18.3200,43.9900,"47,886.3200",0.5479,"87,402.5600",2157,1639,215734,21663,8665,178,True


,event_date,product_id,product_name,product_category,unit_cost,list_price,selling_price,discount_percent,product_views,add_to_cart_events,checkout_started_events,purchases,units_sold,revenue,gross_margin,inventory_level,stockout_flag,price_increase_occurred
0,2024-01-01,P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,45.0400,15.0000,184,22,8,0,0,0.0000,0.0000,759,False,False
1,2024-01-02,P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,39.7400,25.0000,233,37,16,0,0,0.0000,0.0000,759,False,False
2,2024-01-03,P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,52.9900,0.0000,211,16,6,1,1,52.9900,34.7700,758,False,False
3,2024-01-04,P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,47.6900,10.0000,217,21,8,1,2,95.3800,58.9400,756,False,False
4,2024-01-05,P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,52.9900,0.0000,257,21,7,0,0,0.0000,0.0000,756,False,False


## Dataset Shapes and Column Types


In [3]:
shape_summary = pd.DataFrame(
    [
        {"dataset": dataset_name, "rows": len(frame), "columns": frame.shape[1]}
        for dataset_name, frame in frames.items()
    ]
)
display(shape_summary)

for dataset_name, frame in frames.items():
    display(
        pd.DataFrame({f"{dataset_name}_column_type": frame.dtypes.astype(str)})
    )


,dataset,rows,columns
0,customers,12500,13
1,transactions,48769,22
2,products,41,15
3,product_events,29971,18


,customers_column_type
customer_id,object
first_purchase_date,datetime64[ns]
customer_region,object
acquisition_channel,object
customer_tenure_days,int64
purchase_frequency,float64
prior_spending,float64
transaction_count,int64
total_spending,float64
average_discount_percent,float64


,transactions_column_type
transaction_id,object
customer_id,object
transaction_date,datetime64[ns]
product_id,object
product_name,object
product_category,object
product_price,float64
unit_cost,float64
list_price,float64
selling_price,float64


,products_column_type
product_id,object
product_name,object
product_category,object
unit_cost,float64
list_price,float64
gross_margin,float64
gross_margin_rate,float64
total_revenue,float64
total_units_sold,int64
total_purchases,int64


,product_events_column_type
event_date,datetime64[ns]
product_id,object
product_name,object
product_category,object
unit_cost,float64
list_price,float64
selling_price,float64
discount_percent,float64
product_views,int64
add_to_cart_events,int64


## Missing Values and Duplicate Records


In [4]:
def missing_report(frame, dataset_name):
    missing = frame.isna().sum().rename("missing_values").to_frame()
    missing["missing_rate"] = missing["missing_values"] / len(frame)
    missing.insert(0, "dataset", dataset_name)
    return missing.reset_index(names="column")


missing_summary = pd.concat(
    [missing_report(frame, dataset_name) for dataset_name, frame in frames.items()],
    ignore_index=True,
)
display(missing_summary[missing_summary["missing_values"] > 0])

product_event_key_duplicates = product_events.duplicated(
    subset=["product_id", "event_date"]
).sum()

duplicate_summary = pd.DataFrame(
    {
        "check": [
            "duplicate customer_id",
            "duplicate transaction_id",
            "duplicate product_id",
            "duplicate product_name",
            "duplicate product_id/event_date",
        ],
        "count": [
            customers["customer_id"].duplicated().sum(),
            transactions["transaction_id"].duplicated().sum(),
            products["product_id"].duplicated().sum(),
            products["product_name"].duplicated().sum(),
            product_event_key_duplicates,
        ],
    }
)
display(duplicate_summary)


,column,dataset,missing_values,missing_rate


,check,count
0,duplicate customer_id,0
1,duplicate transaction_id,0
2,duplicate product_id,0
3,duplicate product_name,0
4,duplicate product_id/event_date,0


## Product ID Consistency Across Files


In [5]:
customer_ids = set(customers["customer_id"])
product_ids = set(products["product_id"])
transaction_product_ids = set(transactions["product_id"])
event_product_ids = set(product_events["product_id"])

transaction_product_identity = transactions.merge(
    products[["product_id", "product_name", "product_category"]],
    on="product_id",
    how="left",
    suffixes=("", "_catalog"),
)
product_event_identity = product_events.merge(
    products[["product_id", "product_name", "product_category"]],
    on="product_id",
    how="left",
    suffixes=("", "_catalog"),
)

product_id_checks = pd.Series(
    {
        "transactions with unknown customer_id": (
            ~transactions["customer_id"].isin(customer_ids)
        ).sum(),
        "transactions with unknown product_id": (
            ~transactions["product_id"].isin(product_ids)
        ).sum(),
        "product events with unknown product_id": (
            ~product_events["product_id"].isin(product_ids)
        ).sum(),
        "products without transactions": len(product_ids - transaction_product_ids),
        "products without product events": len(product_ids - event_product_ids),
        "transaction product_name mismatch": (
            transaction_product_identity["product_name"]
            != transaction_product_identity["product_name_catalog"]
        ).sum(),
        "transaction product_category mismatch": (
            transaction_product_identity["product_category"]
            != transaction_product_identity["product_category_catalog"]
        ).sum(),
        "product event product_name mismatch": (
            product_event_identity["product_name"]
            != product_event_identity["product_name_catalog"]
        ).sum(),
        "product event product_category mismatch": (
            product_event_identity["product_category"]
            != product_event_identity["product_category_catalog"]
        ).sum(),
    },
    name="flagged_rows",
)

display(product_id_checks.to_frame())

product_coverage = pd.DataFrame(
    {
        "products": [len(products)],
        "products_in_transactions": [transactions["product_id"].nunique()],
        "products_in_events": [product_events["product_id"].nunique()],
        "event_days_per_product_min": [
            product_events.groupby("product_id")["event_date"].nunique().min()
        ],
        "event_days_per_product_max": [
            product_events.groupby("product_id")["event_date"].nunique().max()
        ],
    }
)
display(product_coverage)


,flagged_rows
transactions with unknown customer_id,0
transactions with unknown product_id,0
product events with unknown product_id,0
products without transactions,0
products without product events,0
transaction product_name mismatch,0
transaction product_category mismatch,0
product event product_name mismatch,0
product event product_category mismatch,0


,products,products_in_transactions,products_in_events,event_days_per_product_min,event_days_per_product_max
0,41,41,41,731,731


## Customer and Transaction Validation


In [6]:
transaction_customer_dates = transactions.merge(
    customers[["customer_id", "first_purchase_date"]],
    on="customer_id",
    how="left",
)
# Allow one-cent currency rounding from multi-unit discounted line items.
MONEY_TOLERANCE = 0.011
gross_order_value = transactions["list_price"] * transactions["quantity"]
expected_order_value = (gross_order_value - transactions["discount_amount"]).round(2)
expected_selling_price = (transactions["order_value"] / transactions["quantity"]).round(2)
expected_gross_margin = (
    transactions["order_value"] - transactions["unit_cost"] * transactions["quantity"]
).round(2)

customer_transaction_checks = pd.Series(
    {
        "missing customer_id": customers["customer_id"].isna().sum(),
        "missing transaction_id": transactions["transaction_id"].isna().sum(),
        "negative customer tenure": (customers["customer_tenure_days"] < 0).sum(),
        "negative purchase frequency": (
            (customers["purchase_frequency"] < 0).sum()
            + (transactions["purchase_frequency"] < 0).sum()
        ),
        "negative prior spending": (
            (customers["prior_spending"] < 0).sum()
            + (transactions["prior_spending"] < 0).sum()
        ),
        "non-positive quantity": (transactions["quantity"] <= 0).sum(),
        "transaction before first purchase": (
            transaction_customer_dates["transaction_date"]
            < transaction_customer_dates["first_purchase_date"]
        ).sum(),
        "customer last purchase before first purchase": (
            customers["last_transaction_date"] < customers["first_purchase_date"]
        ).sum(),
        "transaction date outside expected range": (
            (transactions["transaction_date"] < EXPECTED_START_DATE)
            | (transactions["transaction_date"] > EXPECTED_END_DATE)
        ).sum(),
        "customer first purchase outside expected range": (
            (customers["first_purchase_date"] < EXPECTED_START_DATE)
            | (customers["first_purchase_date"] > EXPECTED_END_DATE)
        ).sum(),
    },
    name="flagged_rows",
)

transaction_economics_checks = pd.Series(
    {
        "negative product price": (transactions["product_price"] < 0).sum(),
        "non-positive unit cost": (transactions["unit_cost"] <= 0).sum(),
        "non-positive list price": (transactions["list_price"] <= 0).sum(),
        "negative selling price": (transactions["selling_price"] < 0).sum(),
        "unit cost above list price": (
            transactions["unit_cost"] > transactions["list_price"]
        ).sum(),
        "selling price above list price": (
            transactions["selling_price"] > transactions["list_price"] + MONEY_TOLERANCE
        ).sum(),
        "negative discount amount": (transactions["discount_amount"] < 0).sum(),
        "invalid discount percent": (
            (transactions["discount_percent"] < 0)
            | (transactions["discount_percent"] > 80)
        ).sum(),
        "negative order value": (transactions["order_value"] < 0).sum(),
        "negative gross margin": (transactions["gross_margin"] < 0).sum(),
        "discount exceeds gross order value": (
            transactions["discount_amount"] > gross_order_value + MONEY_TOLERANCE
        ).sum(),
        "order value arithmetic mismatch": (
            (transactions["order_value"] - expected_order_value).abs() > MONEY_TOLERANCE
        ).sum(),
        "selling price arithmetic mismatch": (
            (transactions["selling_price"] - expected_selling_price).abs() > MONEY_TOLERANCE
        ).sum(),
        "gross margin arithmetic mismatch": (
            (transactions["gross_margin"] - expected_gross_margin).abs() > MONEY_TOLERANCE
        ).sum(),
    },
    name="flagged_rows",
)

display(customer_transaction_checks.to_frame())
display(transaction_economics_checks.to_frame())


,flagged_rows
missing customer_id,0
missing transaction_id,0
negative customer tenure,0
negative purchase frequency,0
negative prior spending,0
non-positive quantity,0
transaction before first purchase,0
customer last purchase before first purchase,0
transaction date outside expected range,0
customer first purchase outside expected range,0


,flagged_rows
negative product price,0
non-positive unit cost,0
non-positive list price,0
negative selling price,0
unit cost above list price,0
selling price above list price,0
negative discount amount,0
invalid discount percent,0
negative order value,0
negative gross margin,0


## Product Pricing, Costs, Discounts, and Margins


In [7]:
product_economics_checks = pd.Series(
    {
        "non-positive product unit cost": (products["unit_cost"] <= 0).sum(),
        "non-positive product list price": (products["list_price"] <= 0).sum(),
        "product unit cost above list price": (
            products["unit_cost"] > products["list_price"]
        ).sum(),
        "negative product gross margin": (products["gross_margin"] < 0).sum(),
        "negative product revenue": (products["total_revenue"] < 0).sum(),
        "invalid gross margin rate": (
            (products["gross_margin_rate"] < 0)
            | (products["gross_margin_rate"] > 1)
        ).sum(),
        "negative event unit cost": (product_events["unit_cost"] < 0).sum(),
        "non-positive event list price": (product_events["list_price"] <= 0).sum(),
        "negative event selling price": (product_events["selling_price"] < 0).sum(),
        "event unit cost above list price": (
            product_events["unit_cost"] > product_events["list_price"]
        ).sum(),
        "invalid event discount percent": (
            (product_events["discount_percent"] < 0)
            | (product_events["discount_percent"] > 80)
        ).sum(),
        "negative event revenue": (product_events["revenue"] < 0).sum(),
        "negative event gross margin": (product_events["gross_margin"] < 0).sum(),
    },
    name="flagged_rows",
)

display(product_economics_checks.to_frame())

economics_summary = pd.concat(
    {
        "transactions": transactions[
            ["unit_cost", "list_price", "selling_price", "discount_percent", "order_value", "gross_margin"]
        ].describe().T,
        "products": products[
            ["unit_cost", "list_price", "gross_margin_rate", "total_revenue", "gross_margin"]
        ].describe().T,
        "product_events": product_events[
            ["list_price", "selling_price", "discount_percent", "revenue", "gross_margin"]
        ].describe().T,
    }
)
display(economics_summary)


,flagged_rows
non-positive product unit cost,0
non-positive product list price,0
product unit cost above list price,0
negative product gross margin,0
negative product revenue,0
invalid gross margin rate,0
negative event unit cost,0
non-positive event list price,0
negative event selling price,0
event unit cost above list price,0


count        mean         std  \
transactions   unit_cost         48,769.0000     20.2027      7.9585   
               list_price        48,769.0000     47.4293     18.0187   
               selling_price     48,769.0000     43.4965     17.1605   
               discount_percent  48,769.0000      8.2834      9.2424   
               order_value       48,769.0000     57.5323     36.1687   
               gross_margin      48,769.0000     30.6716     20.4846   
products       unit_cost             41.0000     20.4580      8.7455   
               list_price            41.0000     47.5022     19.3263   
               gross_margin_rate     41.0000      0.5294      0.0615   
               total_revenue         41.0000 68,433.9515 26,745.7048   
               gross_margin          41.0000 36,483.5056 15,675.5919   
product_events list_price        29,971.0000     47.9005     19.3023   
               selling_price     29,971.0000     44.6658     18.5365   
               discount_percent  29,971.0000      6.7119      7.8910   
               revenue           29,971.0000     93.6169    105.4643   
               gross_margin      29,971.0000     49.9090     57.6873   

                                         min         25%         50%  \
transactions   unit_cost              7.2900     14.3900     17.7200   
               list_price            17.9900     33.9900     41.9900   
               selling_price         12.5900     30.9900     39.8900   
               discount_percent       0.0000      0.0000     10.0000   
               order_value           12.5900     33.5900     48.9900   
               gross_margin           4.5200     17.7400     25.6000   
products       unit_cost              7.2900     14.3900     17.7200   
               list_price            17.9900     33.9900     41.9900   
               gross_margin_rate      0.4400      0.4715      0.5200   
               total_revenue     18,715.3700 49,188.0000 71,492.0500   
               gross_margin       9,915.0900 25,887.0300 34,518.7600   
product_events list_price            17.9900     33.9900     41.9900   
               selling_price         12.5900     30.9900     39.9900   
               discount_percent       0.0000      0.0000      3.7500   
               revenue                0.0000      0.0000     62.8800   
               gross_margin           0.0000      0.0000     33.4900   

                                         75%          max  
transactions   unit_cost             25.9800      45.1900  
               list_price            58.9900     104.9900  
               selling_price         54.9900     104.9900  
               discount_percent      15.0000      30.0000  
               order_value           67.9800     379.9600  
               gross_margin          37.2600     226.4800  
products       unit_cost             25.9800      45.1900  
               list_price            58.9900      93.9900  
               gross_margin_rate      0.5897       0.6274  
               total_revenue     88,715.9400 119,532.0300  
               gross_margin      48,080.4400  67,849.3200  
product_events list_price            58.9900     104.9900  
               selling_price         55.9900     104.9900  
               discount_percent      11.6700      30.0000  
               revenue              136.7600   1,142.8700  
               gross_margin          72.0000     580.5200

## Product-Level Aggregate Consistency


In [8]:
product_event_totals = product_events.groupby("product_id").agg(
    event_revenue=("revenue", "sum"),
    event_gross_margin=("gross_margin", "sum"),
    event_units_sold=("units_sold", "sum"),
    event_purchases=("purchases", "sum"),
    event_product_views=("product_views", "sum"),
    event_add_to_cart_events=("add_to_cart_events", "sum"),
    event_checkout_started_events=("checkout_started_events", "sum"),
    latest_inventory_level=("inventory_level", "last"),
    any_stockout=("stockout_flag", "max"),
)
product_total_check = products.set_index("product_id").join(product_event_totals)

aggregate_consistency_checks = pd.Series(
    {
        "product revenue does not match events": (
            (product_total_check["total_revenue"] - product_total_check["event_revenue"]).abs()
            > MONEY_TOLERANCE
        ).sum(),
        "product gross margin does not match events": (
            (
                product_total_check["gross_margin"]
                - product_total_check["event_gross_margin"]
            ).abs()
            > MONEY_TOLERANCE
        ).sum(),
        "product units sold do not match events": (
            product_total_check["total_units_sold"]
            != product_total_check["event_units_sold"]
        ).sum(),
        "product purchases do not match events": (
            product_total_check["total_purchases"]
            != product_total_check["event_purchases"]
        ).sum(),
        "product views do not match events": (
            product_total_check["product_views"]
            != product_total_check["event_product_views"]
        ).sum(),
        "add-to-cart totals do not match events": (
            product_total_check["add_to_cart_events"]
            != product_total_check["event_add_to_cart_events"]
        ).sum(),
        "checkout-start totals do not match events": (
            product_total_check["checkout_started_events"]
            != product_total_check["event_checkout_started_events"]
        ).sum(),
        "inventory level does not match latest event": (
            product_total_check["inventory_level"]
            != product_total_check["latest_inventory_level"]
        ).sum(),
        "stockout flag does not match any event stockout": (
            product_total_check["stockout_flag"]
            != product_total_check["any_stockout"]
        ).sum(),
    },
    name="flagged_products",
)

display(aggregate_consistency_checks.to_frame())
display(product_total_check.head())


,flagged_products
product revenue does not match events,0
product gross margin does not match events,0
product units sold do not match events,0
product purchases do not match events,0
product views do not match events,0
add-to-cart totals do not match events,0
checkout-start totals do not match events,0
inventory level does not match latest event,0
stockout flag does not match any event stockout,0


,product_name,product_category,unit_cost,list_price,gross_margin,gross_margin_rate,total_revenue,total_units_sold,total_purchases,product_views,add_to_cart_events,checkout_started_events,inventory_level,stockout_flag,event_revenue,event_gross_margin,event_units_sold,event_purchases,event_product_views,event_add_to_cart_events,event_checkout_started_events,latest_inventory_level,any_stockout
product_id,,,,,,,,,,,,,,,,,,,,,,,
P0001,Apex Sculpt Leggings,leggings,18.2200,52.9900,"63,143.2500",0.6266,"100,767.5500",2065,1507,195059,19763,7967,79,True,"100,767.5500","63,143.2500",2065,1507,195059,19763,7967,79,True
P0002,Core Flex Leggings,leggings,25.2000,53.9900,"38,464.4400",0.4918,"78,204.8400",1577,1214,170860,16926,6741,408,False,"78,204.8400","38,464.4400",1577,1214,170860,16926,6741,408,False
P0003,Motion Seamless Leggings,leggings,20.1900,54.9900,"58,885.7500",0.5976,"98,538.9100",1964,1464,205376,20769,8377,424,False,"98,538.9100","58,885.7500",1964,1464,205376,20769,8377,424,False
P0004,Lift Training Leggings,leggings,27.0700,55.9900,"53,752.4300",0.4715,"114,010.2500",2226,1681,208741,20888,8335,89,True,"114,010.2500","53,752.4300",2226,1681,208741,20888,8335,89,True
P0005,Contour High-Rise Leggings,leggings,18.3200,43.9900,"47,886.3200",0.5479,"87,402.5600",2157,1639,215734,21663,8665,178,True,"87,402.5600","47,886.3200",2157,1639,215734,21663,8665,178,True


## Transaction Date Distribution


In [9]:
monthly_transactions = (
    transactions.set_index("transaction_date")
    .resample("MS")
    .size()
    .rename("transaction_count")
    .to_frame()
)
display(monthly_transactions.T)
display(monthly_transactions.style.bar(subset=["transaction_count"], color="#4C78A8"))

if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(10, 4))
    monthly_transactions["transaction_count"].plot(ax=ax, marker="o")
    ax.set_title("Monthly Transaction Count")
    ax.set_xlabel("Transaction month")
    ax.set_ylabel("Transactions")
    plt.tight_layout()
    plt.show()


transaction_date,2024-01-01,2024-02-01,2024-03-01,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01
transaction_count,568,637,825,924,1093,1144,1357,1376,1554,1711,2036,2175,1925,1880,2216,2379,2506,2783,2995,2963,3013,3229,3825,3655


,transaction_count
transaction_date,
2024-01-01 00:00:00,568
2024-02-01 00:00:00,637
2024-03-01 00:00:00,825
2024-04-01 00:00:00,924
2024-05-01 00:00:00,1093
2024-06-01 00:00:00,1144
2024-07-01 00:00:00,1357
2024-08-01 00:00:00,1376
2024-09-01 00:00:00,1554


## Customer Transaction Counts


In [10]:
observed_transaction_counts = transactions.groupby("customer_id").size().rename(
    "observed_transaction_count"
)
customer_transaction_check = customers.set_index("customer_id")[["transaction_count"]].join(
    observed_transaction_counts,
    how="left",
)
customer_transaction_check["observed_transaction_count"] = customer_transaction_check[
    "observed_transaction_count"
].fillna(0).astype(int)
customer_transaction_check["count_mismatch"] = (
    customer_transaction_check["transaction_count"]
    != customer_transaction_check["observed_transaction_count"]
)

display(customer_transaction_check.describe().T)
display(
    pd.DataFrame(
        {
            "customers_without_transactions": [
                (customer_transaction_check["observed_transaction_count"] == 0).sum()
            ],
            "transaction_count_mismatches": [
                customer_transaction_check["count_mismatch"].sum()
            ],
        }
    )
)

transaction_count_distribution = (
    customer_transaction_check["observed_transaction_count"]
    .clip(upper=12)
    .value_counts()
    .sort_index()
    .rename_axis("transactions_per_customer_capped")
    .to_frame("customers")
)
display(transaction_count_distribution.style.bar(subset=["customers"], color="#59A14F"))

if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 4))
    customer_transaction_check["observed_transaction_count"].clip(upper=12).plot(
        kind="hist",
        bins=range(1, 14),
        ax=ax,
    )
    ax.set_title("Customer Transaction Counts, Capped at 12")
    ax.set_xlabel("Transactions per customer")
    ax.set_ylabel("Customers")
    plt.tight_layout()
    plt.show()


,count,mean,std,min,25%,50%,75%,max
transaction_count,"12,500.0000",3.9015,2.6335,1.0000,2.0000,3.0000,5.0000,18.0000
observed_transaction_count,"12,500.0000",3.9015,2.6335,1.0000,2.0000,3.0000,5.0000,18.0000


,customers_without_transactions,transaction_count_mismatches
0,0,0


,customers
transactions_per_customer_capped,
1,2074
2,2501
3,2157
4,1711
5,1285
6,892
7,611
8,410
9,302


## Order Value Distribution


In [11]:
order_value_summary = transactions["order_value"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)
display(order_value_summary.to_frame())

order_value_bins = pd.cut(
    transactions["order_value"],
    bins=[0, 25, 50, 75, 100, 150, 200, transactions["order_value"].max() + 1],
    include_lowest=True,
)
order_value_distribution = order_value_bins.value_counts().sort_index().to_frame("transactions")
display(order_value_distribution.style.bar(subset=["transactions"], color="#F28E2B"))

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    transactions["order_value"].plot(kind="hist", bins=50, ax=axes[0])
    axes[0].set_title("Order Value Distribution")
    axes[0].set_xlabel("Order value")
    axes[0].set_ylabel("Transactions")

    axes[1].boxplot(transactions["order_value"], vert=False)
    axes[1].set_title("Order Value Spread")
    axes[1].set_xlabel("Order value")
    plt.tight_layout()
    plt.show()


,order_value
count,"48,769.0000"
mean,57.5323
std,36.1687
min,12.5900
1%,17.5900
5%,21.9900
25%,33.5900
50%,48.9900
75%,67.9800
95%,131.9600


,transactions
order_value,
"(-0.001, 25.0]",4078
"(25.0, 50.0]",21338
"(50.0, 75.0]",13554
"(75.0, 100.0]",4752
"(100.0, 150.0]",3509
"(150.0, 200.0]",1096
"(200.0, 380.96]",442


## Churn Distribution and Price-Change Frequency


In [12]:
churn_distribution = customers["churned"].value_counts(dropna=False).rename_axis(
    "churned"
).to_frame("customers")
churn_distribution["share"] = churn_distribution["customers"] / len(customers)

price_change_distribution = transactions[
    "price_increase_occurred"
].value_counts(dropna=False).rename_axis("price_increase_occurred").to_frame(
    "transactions"
)
price_change_distribution["share"] = (
    price_change_distribution["transactions"] / len(transactions)
)

product_event_price_change_distribution = product_events[
    "price_increase_occurred"
].value_counts(dropna=False).rename_axis("price_increase_occurred").to_frame(
    "product_events"
)
product_event_price_change_distribution["share"] = (
    product_event_price_change_distribution["product_events"] / len(product_events)
)

display(churn_distribution.style.bar(subset=["share"], color="#E15759"))
display(price_change_distribution.style.bar(subset=["share"], color="#76B7B2"))
display(product_event_price_change_distribution.style.bar(subset=["share"], color="#9C755F"))

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    churn_distribution["share"].sort_index().plot(kind="bar", ax=axes[0])
    axes[0].set_title("Customer Churn Share")
    axes[0].set_xlabel("Churned")
    axes[0].set_ylabel("Share")

    price_change_distribution["share"].sort_index().plot(kind="bar", ax=axes[1])
    axes[1].set_title("Transaction Price-Change Share")
    axes[1].set_xlabel("Price increase occurred")
    axes[1].set_ylabel("Share")

    product_event_price_change_distribution["share"].sort_index().plot(kind="bar", ax=axes[2])
    axes[2].set_title("Product-Day Price-Change Share")
    axes[2].set_xlabel("Price increase occurred")
    axes[2].set_ylabel("Share")
    plt.tight_layout()
    plt.show()


,customers,share
churned,,
False,11590,0.927200
True,910,0.072800


,transactions,share
price_increase_occurred,,
False,44466,0.911768
True,4303,0.088232


,product_events,share
price_increase_occurred,,
False,26804,0.894331
True,3167,0.105669


## Conversion Funnel Validation

Funnel rates are calculated with the correct sequential denominator at each stage: add-to-cart divided by product views, checkout-start divided by add-to-cart, and purchase-after-checkout divided by checkout-start.


In [13]:
def safe_rate(numerator, denominator):
    return numerator / denominator if denominator else pd.NA


funnel_totals = {
    "product_views": int(product_events["product_views"].sum()),
    "add_to_cart_events": int(product_events["add_to_cart_events"].sum()),
    "checkout_started_events": int(product_events["checkout_started_events"].sum()),
    "purchases": int(product_events["purchases"].sum()),
}

funnel_rates = pd.DataFrame(
    [
        {
            "stage": "Add to cart",
            "numerator": "add_to_cart_events",
            "denominator": "product_views",
            "numerator_value": funnel_totals["add_to_cart_events"],
            "denominator_value": funnel_totals["product_views"],
            "rate": safe_rate(
                funnel_totals["add_to_cart_events"],
                funnel_totals["product_views"],
            ),
        },
        {
            "stage": "Checkout started",
            "numerator": "checkout_started_events",
            "denominator": "add_to_cart_events",
            "numerator_value": funnel_totals["checkout_started_events"],
            "denominator_value": funnel_totals["add_to_cart_events"],
            "rate": safe_rate(
                funnel_totals["checkout_started_events"],
                funnel_totals["add_to_cart_events"],
            ),
        },
        {
            "stage": "Purchase after checkout",
            "numerator": "purchases",
            "denominator": "checkout_started_events",
            "numerator_value": funnel_totals["purchases"],
            "denominator_value": funnel_totals["checkout_started_events"],
            "rate": safe_rate(
                funnel_totals["purchases"],
                funnel_totals["checkout_started_events"],
            ),
        },
        {
            "stage": "View to purchase",
            "numerator": "purchases",
            "denominator": "product_views",
            "numerator_value": funnel_totals["purchases"],
            "denominator_value": funnel_totals["product_views"],
            "rate": safe_rate(funnel_totals["purchases"], funnel_totals["product_views"]),
        },
    ]
)

display(pd.DataFrame(funnel_totals, index=["total"]))
display(funnel_rates)

funnel_validation_checks = pd.Series(
    {
        "negative product views": (product_events["product_views"] < 0).sum(),
        "negative add-to-cart events": (product_events["add_to_cart_events"] < 0).sum(),
        "negative checkout-start events": (product_events["checkout_started_events"] < 0).sum(),
        "negative purchases": (product_events["purchases"] < 0).sum(),
        "add-to-cart exceeds views": (
            product_events["add_to_cart_events"] > product_events["product_views"]
        ).sum(),
        "checkout-start exceeds add-to-cart": (
            product_events["checkout_started_events"]
            > product_events["add_to_cart_events"]
        ).sum(),
        "purchases exceed checkout-start": (
            product_events["purchases"]
            > product_events["checkout_started_events"]
        ).sum(),
        "units sold below purchases": (
            (product_events["purchases"] > 0)
            & (product_events["units_sold"] < product_events["purchases"])
        ).sum(),
        "funnel activity with zero views": (
            (product_events["product_views"] == 0)
            & (
                product_events[
                    ["add_to_cart_events", "checkout_started_events", "purchases"]
                ].sum(axis=1)
                > 0
            )
        ).sum(),
    },
    name="flagged_rows",
)

display(funnel_validation_checks.to_frame())

if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 4))
    funnel_rates.set_index("stage")["rate"].plot(kind="bar", ax=ax, color="#4E79A7")
    ax.set_title("Sequential Funnel Rates")
    ax.set_xlabel("Funnel stage")
    ax.set_ylabel("Rate")
    ax.set_ylim(0, max(0.5, funnel_rates["rate"].max() * 1.2))
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()


,product_views,add_to_cart_events,checkout_started_events,purchases
total,6808066,678053,270968,48769


,stage,numerator,denominator,numerator_value,denominator_value,rate
0,Add to cart,add_to_cart_events,product_views,678053,6808066,0.0996
1,Checkout started,checkout_started_events,add_to_cart_events,270968,678053,0.3996
2,Purchase after checkout,purchases,checkout_started_events,48769,270968,0.1800
3,View to purchase,purchases,product_views,48769,6808066,0.0072


,flagged_rows
negative product views,0
negative add-to-cart events,0
negative checkout-start events,0
negative purchases,0
add-to-cart exceeds views,0
checkout-start exceeds add-to-cart,0
purchases exceed checkout-start,0
units sold below purchases,0
funnel activity with zero views,0


## Product Views, Movement, and Price-Variation Support


In [14]:
product_funnel = products.assign(
    add_to_cart_rate=lambda frame: frame["add_to_cart_events"] / frame["product_views"],
    checkout_start_rate=lambda frame: frame["checkout_started_events"] / frame["add_to_cart_events"],
    purchase_after_checkout_rate=lambda frame: frame["total_purchases"] / frame["checkout_started_events"],
    view_to_purchase_rate=lambda frame: frame["total_purchases"] / frame["product_views"],
    revenue_per_view=lambda frame: frame["total_revenue"] / frame["product_views"],
)

price_variation_support = product_events.groupby("product_id").agg(
    product_name=("product_name", "first"),
    unique_selling_prices=("selling_price", "nunique"),
    min_selling_price=("selling_price", "min"),
    max_selling_price=("selling_price", "max"),
    price_increase_day_share=("price_increase_occurred", "mean"),
    views=("product_views", "sum"),
    purchases=("purchases", "sum"),
)
price_variation_support["selling_price_range"] = (
    price_variation_support["max_selling_price"]
    - price_variation_support["min_selling_price"]
)

movement_summary = product_funnel[
    [
        "total_revenue",
        "gross_margin",
        "gross_margin_rate",
        "total_units_sold",
        "total_purchases",
        "product_views",
        "add_to_cart_rate",
        "checkout_start_rate",
        "purchase_after_checkout_rate",
        "view_to_purchase_rate",
        "revenue_per_view",
    ]
].describe().T

display(movement_summary)
display(
    product_funnel.sort_values("total_revenue", ascending=False)[
        ["product_id", "product_name", "product_category", "total_revenue", "gross_margin", "total_units_sold", "view_to_purchase_rate"]
    ].head(10)
)
display(
    product_funnel.sort_values("total_purchases")[
        ["product_id", "product_name", "product_category", "total_purchases", "product_views", "view_to_purchase_rate", "inventory_level", "stockout_flag"]
    ].head(10)
)
display(price_variation_support.describe().T)

display(
    pd.DataFrame(
        {
            "products_with_multiple_selling_prices": [
                (price_variation_support["unique_selling_prices"] > 1).sum()
            ],
            "products_with_price_increase_days": [
                (price_variation_support["price_increase_day_share"] > 0).sum()
            ],
            "products_with_positive_sales": [(products["total_purchases"] > 0).sum()],
        }
    )
)

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    product_funnel["product_views"].plot(kind="hist", bins=20, ax=axes[0])
    axes[0].set_title("Product View Distribution")
    axes[0].set_xlabel("Views per product")

    product_funnel["total_units_sold"].plot(kind="hist", bins=20, ax=axes[1], color="#59A14F")
    axes[1].set_title("Product Units Sold Distribution")
    axes[1].set_xlabel("Units sold per product")
    plt.tight_layout()
    plt.show()


,count,mean,std,min,25%,50%,75%,max
total_revenue,41.0000,"68,433.9515","26,745.7048","18,715.3700","49,188.0000","71,492.0500","88,715.9400","119,532.0300"
gross_margin,41.0000,"36,483.5056","15,675.5919","9,915.0900","25,887.0300","34,518.7600","48,080.4400","67,849.3200"
gross_margin_rate,41.0000,0.5294,0.0615,0.4400,0.4715,0.5200,0.5897,0.6274
total_units_sold,41.0000,"1,581.3415",313.6140,"1,030.0000","1,353.0000","1,554.0000","1,797.0000","2,226.0000"
total_purchases,41.0000,"1,189.4878",237.2250,769.0000,"1,049.0000","1,170.0000","1,368.0000","1,681.0000"
product_views,41.0000,"166,050.3902","25,126.0358","120,900.0000","149,640.0000","162,033.0000","184,398.0000","215,734.0000"
add_to_cart_rate,41.0000,0.0995,0.0010,0.0970,0.0989,0.0993,0.1002,0.1013
checkout_start_rate,41.0000,0.3995,0.0013,0.3972,0.3985,0.3997,0.4000,0.4033
purchase_after_checkout_rate,41.0000,0.1789,0.0093,0.1589,0.1744,0.1783,0.1854,0.2017
view_to_purchase_rate,41.0000,0.0071,0.0004,0.0062,0.0069,0.0071,0.0074,0.0081


,product_id,product_name,product_category,total_revenue,gross_margin,total_units_sold,view_to_purchase_rate
35,P0036,Lift Zip Jacket,outerwear,"119,532.0300","67,849.3200",1383,0.0070
3,P0004,Lift Training Leggings,leggings,"114,010.2500","53,752.4300",2226,0.0081
34,P0035,Motion Lightweight Jacket,outerwear,"104,910.4100","47,383.5400",1273,0.0068
33,P0034,Core Puffer Vest,outerwear,"100,938.3500","59,685.3800",1353,0.0070
0,P0001,Apex Sculpt Leggings,leggings,"100,767.5500","63,143.2500",2065,0.0077
2,P0003,Motion Seamless Leggings,leggings,"98,538.9100","58,885.7500",1964,0.0071
19,P0020,Motion Zip Hoodie,hoodies,"98,421.5800","53,226.4800",1597,0.0075
20,P0021,Lift Oversized Hoodie,hoodies,"98,200.8700","57,910.5700",1509,0.0069
25,P0026,Lift Tapered Joggers,joggers,"97,414.6000","45,912.5800",1797,0.0072
22,P0023,Apex Slim Joggers,joggers,"89,309.9600","54,135.2600",1710,0.0074


,product_id,product_name,product_category,total_purchases,product_views,view_to_purchase_rate,inventory_level,stockout_flag
32,P0033,Apex Training Jacket,outerwear,769,120900,0.0064,348,False
39,P0040,Lift Wrist Wraps,accessories,791,126883,0.0062,99,True
31,P0032,Tempo Woven Shorts,shorts,816,126674,0.0064,817,False
38,P0039,Motion Duffel Bag,accessories,840,131750,0.0064,226,False
30,P0031,Lift Hybrid Shorts,shorts,861,129444,0.0067,263,False
34,P0035,Motion Lightweight Jacket,outerwear,928,136353,0.0068,751,False
24,P0025,Motion Training Joggers,joggers,972,146137,0.0067,426,False
36,P0037,Core Crew Socks,accessories,1002,150796,0.0066,119,True
33,P0034,Core Puffer Vest,outerwear,1016,145066,0.0070,401,False
27,P0028,Apex Training Shorts,shorts,1025,139270,0.0074,56,True


,count,mean,std,min,25%,50%,75%,max
unique_selling_prices,41.0000,86.9024,16.4070,53.0000,73.0000,87.0000,98.0000,124.0000
min_selling_price,41.0000,33.2485,13.5284,12.5900,23.7900,29.3900,41.2900,65.7900
max_selling_price,41.0000,53.0388,21.6309,19.9900,37.9900,46.9900,65.9900,104.9900
price_increase_day_share,41.0000,0.1057,0.0383,0.0410,0.0670,0.1135,0.1395,0.1655
views,41.0000,"166,050.3902","25,126.0358","120,900.0000","149,640.0000","162,033.0000","184,398.0000","215,734.0000"
purchases,41.0000,"1,189.4878",237.2250,769.0000,"1,049.0000","1,170.0000","1,368.0000","1,681.0000"
selling_price_range,41.0000,19.7902,8.1087,7.4000,14.2000,17.6000,24.7000,39.2000


,products_with_multiple_selling_prices,products_with_price_increase_days,products_with_positive_sales
0,41,41,41


## Inventory Levels and Stockout Rates


In [15]:
inventory_checks = pd.Series(
    {
        "negative event inventory level": (product_events["inventory_level"] < 0).sum(),
        "missing event stockout flag": product_events["stockout_flag"].isna().sum(),
        "missing product stockout flag": products["stockout_flag"].isna().sum(),
        "products with no final inventory": (products["inventory_level"].isna()).sum(),
        "negative product inventory level": (products["inventory_level"] < 0).sum(),
    },
    name="flagged_rows",
)

display(inventory_checks.to_frame())

stockout_summary = pd.DataFrame(
    {
        "metric": [
            "product-day stockout rate",
            "products with any stockout",
            "average final inventory",
            "median final inventory",
        ],
        "value": [
            product_events["stockout_flag"].mean(),
            products["stockout_flag"].mean(),
            products["inventory_level"].mean(),
            products["inventory_level"].median(),
        ],
    }
)
display(stockout_summary)

stockout_by_category = product_events.groupby("product_category").agg(
    product_days=("stockout_flag", "size"),
    stockout_rate=("stockout_flag", "mean"),
    average_inventory=("inventory_level", "mean"),
    purchases=("purchases", "sum"),
).sort_values("stockout_rate", ascending=False)
display(stockout_by_category)

if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(9, 4))
    stockout_by_category["stockout_rate"].plot(kind="bar", ax=ax, color="#E15759")
    ax.set_title("Stockout Rate by Product Category")
    ax.set_xlabel("Product category")
    ax.set_ylabel("Stockout rate")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


,flagged_rows
negative event inventory level,0
missing event stockout flag,0
missing product stockout flag,0
products with no final inventory,0
negative product inventory level,0


,metric,value
0,product-day stockout rate,0.0563
1,products with any stockout,0.4878
2,average final inventory,306.2195
3,median final inventory,180.0000


,product_days,stockout_rate,average_inventory,purchases
product_category,,,,
joggers,3655,0.1045,616.9669,5857
hoodies,3655,0.0824,724.5253,5680
shorts,3655,0.0807,630.5231,4989
leggings,4386,0.0673,771.4223,8701
sports_bras,3655,0.0454,734.9409,6679
accessories,3655,0.0317,495.5850,4788
outerwear,2924,0.0263,713.3789,3762
training_tops,4386,0.0128,882.7027,8313


## Basic Summary Statistics


In [16]:
customer_numeric_columns = [
    "customer_tenure_days",
    "purchase_frequency",
    "prior_spending",
    "transaction_count",
    "total_spending",
    "average_discount_percent",
]
transaction_numeric_columns = [
    "product_price",
    "unit_cost",
    "list_price",
    "selling_price",
    "discount_amount",
    "discount_percent",
    "quantity",
    "order_value",
    "gross_margin",
    "customer_tenure_days",
    "purchase_frequency",
    "prior_spending",
]
product_numeric_columns = [
    "unit_cost",
    "list_price",
    "gross_margin",
    "gross_margin_rate",
    "total_revenue",
    "total_units_sold",
    "total_purchases",
    "product_views",
    "add_to_cart_events",
    "checkout_started_events",
    "inventory_level",
]
product_event_numeric_columns = [
    "unit_cost",
    "list_price",
    "selling_price",
    "discount_percent",
    "product_views",
    "add_to_cart_events",
    "checkout_started_events",
    "purchases",
    "units_sold",
    "revenue",
    "gross_margin",
    "inventory_level",
]

display(customers[customer_numeric_columns].describe().T)
display(transactions[transaction_numeric_columns].describe().T)
display(products[product_numeric_columns].describe().T)
display(product_events[product_event_numeric_columns].describe().T)

display(customers["customer_region"].value_counts(normalize=True).to_frame("customer_share"))
display(customers["acquisition_channel"].value_counts(normalize=True).to_frame("customer_share"))
display(transactions["product_category"].value_counts(normalize=True).to_frame("transaction_share"))
display(products["product_category"].value_counts().to_frame("products"))


,count,mean,std,min,25%,50%,75%,max
customer_tenure_days,"12,500.0000",379.3454,203.9844,30.0000,201.0000,377.0000,560.0000,730.0000
purchase_frequency,"12,500.0000",4.1507,2.2336,0.5020,2.5090,3.8290,5.3450,16.0000
prior_spending,"12,500.0000",224.4634,167.1391,14.3900,98.9800,183.5050,304.9525,"1,224.8900"
transaction_count,"12,500.0000",3.9015,2.6335,1.0000,2.0000,3.0000,5.0000,18.0000
total_spending,"12,500.0000",224.4634,167.1391,14.3900,98.9800,183.5050,304.9525,"1,224.8900"
average_discount_percent,"12,500.0000",8.1624,5.9209,0.0000,3.7500,8.0000,11.6700,30.0000


,count,mean,std,min,25%,50%,75%,max
product_price,"48,769.0000",47.4293,18.0187,17.9900,33.9900,41.9900,58.9900,104.9900
unit_cost,"48,769.0000",20.2027,7.9585,7.2900,14.3900,17.7200,25.9800,45.1900
list_price,"48,769.0000",47.4293,18.0187,17.9900,33.9900,41.9900,58.9900,104.9900
selling_price,"48,769.0000",43.4965,17.1605,12.5900,30.9900,39.8900,54.9900,104.9900
discount_amount,"48,769.0000",5.5251,8.3448,0.0000,0.0000,2.2000,8.4000,93.9900
discount_percent,"48,769.0000",8.2834,9.2424,0.0000,0.0000,10.0000,15.0000,30.0000
quantity,"48,769.0000",1.3294,0.6287,1.0000,1.0000,1.0000,2.0000,4.0000
order_value,"48,769.0000",57.5323,36.1687,12.5900,33.5900,48.9900,67.9800,379.9600
gross_margin,"48,769.0000",30.6716,20.4846,4.5200,17.7400,25.6000,37.2600,226.4800
customer_tenure_days,"48,769.0000",191.0642,185.3892,0.0000,0.0000,146.0000,319.0000,728.0000


,count,mean,std,min,25%,50%,75%,max
unit_cost,41.0000,20.4580,8.7455,7.2900,14.3900,17.7200,25.9800,45.1900
list_price,41.0000,47.5022,19.3263,17.9900,33.9900,41.9900,58.9900,93.9900
gross_margin,41.0000,"36,483.5056","15,675.5919","9,915.0900","25,887.0300","34,518.7600","48,080.4400","67,849.3200"
gross_margin_rate,41.0000,0.5294,0.0615,0.4400,0.4715,0.5200,0.5897,0.6274
total_revenue,41.0000,"68,433.9515","26,745.7048","18,715.3700","49,188.0000","71,492.0500","88,715.9400","119,532.0300"
total_units_sold,41.0000,"1,581.3415",313.6140,"1,030.0000","1,353.0000","1,554.0000","1,797.0000","2,226.0000"
total_purchases,41.0000,"1,189.4878",237.2250,769.0000,"1,049.0000","1,170.0000","1,368.0000","1,681.0000"
product_views,41.0000,"166,050.3902","25,126.0358","120,900.0000","149,640.0000","162,033.0000","184,398.0000","215,734.0000"
add_to_cart_events,41.0000,"16,537.8780","2,618.1919","11,723.0000","14,763.0000","16,024.0000","18,519.0000","21,663.0000"
checkout_started_events,41.0000,"6,608.9756","1,055.4434","4,696.0000","5,904.0000","6,408.0000","7,413.0000","8,665.0000"


,count,mean,std,min,25%,50%,75%,max
unit_cost,"29,971.0000",20.4580,8.6383,7.2900,14.3900,17.7200,25.9800,45.1900
list_price,"29,971.0000",47.9005,19.3023,17.9900,33.9900,41.9900,58.9900,104.9900
selling_price,"29,971.0000",44.6658,18.5365,12.5900,30.9900,39.9900,55.9900,104.9900
discount_percent,"29,971.0000",6.7119,7.8910,0.0000,0.0000,3.7500,11.6700,30.0000
product_views,"29,971.0000",227.1551,71.0432,55.0000,180.0000,216.0000,261.0000,813.0000
add_to_cart_events,"29,971.0000",22.6236,9.3454,5.0000,16.0000,21.0000,28.0000,81.0000
checkout_started_events,"29,971.0000",9.0410,4.0399,2.0000,6.0000,8.0000,11.0000,34.0000
purchases,"29,971.0000",1.6272,1.5271,0.0000,0.0000,1.0000,2.0000,11.0000
units_sold,"29,971.0000",2.1633,2.1921,0.0000,0.0000,2.0000,3.0000,19.0000
revenue,"29,971.0000",93.6169,105.4643,0.0000,0.0000,62.8800,136.7600,"1,142.8700"


,customer_share
customer_region,
uk,0.3441
north_america,0.3043
europe,0.1873
asia_pacific,0.0974
rest_of_world,0.0669


,customer_share
acquisition_channel,
paid_social,0.2852
influencer,0.1961
organic_search,0.1747
direct,0.1636
email,0.1038
affiliate,0.0766


,transaction_share
product_category,
leggings,0.1784
training_tops,0.1705
sports_bras,0.1370
joggers,0.1201
hoodies,0.1165
shorts,0.1023
accessories,0.0982
outerwear,0.0771


,products
product_category,
leggings,6
training_tops,6
sports_bras,5
hoodies,5
joggers,5
shorts,5
accessories,5
outerwear,4


## Validation Findings


In [17]:
issues = []
suspicious_patterns = []
assumption_notes = []

expected_event_rows = len(products) * (
    (EXPECTED_END_DATE - EXPECTED_START_DATE).days + 1
)

average_order_value = transactions["order_value"].mean()
churn_rate = customers["churned"].mean()
price_change_frequency = transactions["price_increase_occurred"].mean()
transactions_per_customer = len(transactions) / len(customers)
gross_margin_rate = transactions["gross_margin"].sum() / transactions["order_value"].sum()
stockout_product_day_rate = product_events["stockout_flag"].mean()
add_to_cart_rate = funnel_rates.loc[funnel_rates["stage"] == "Add to cart", "rate"].iloc[0]
checkout_start_rate = funnel_rates.loc[funnel_rates["stage"] == "Checkout started", "rate"].iloc[0]
purchase_after_checkout_rate = funnel_rates.loc[
    funnel_rates["stage"] == "Purchase after checkout", "rate"
].iloc[0]

customer_checks = pd.Series(
    {
        "missing customer_id": customers["customer_id"].isna().sum(),
        "duplicate customer_id": customers["customer_id"].duplicated().sum(),
        "missing customer fields": customers.isna().sum().sum(),
        "negative customer tenure": (customers["customer_tenure_days"] < 0).sum(),
        "negative purchase frequency": (customers["purchase_frequency"] < 0).sum(),
        "negative prior spending": (customers["prior_spending"] < 0).sum(),
        "negative transaction count": (customers["transaction_count"] < 0).sum(),
        "negative total spending": (customers["total_spending"] < 0).sum(),
        "invalid first purchase date": (
            (customers["first_purchase_date"] < EXPECTED_START_DATE)
            | (customers["first_purchase_date"] > EXPECTED_END_DATE)
        ).sum(),
        "last purchase before first purchase": (
            customers["last_transaction_date"] < customers["first_purchase_date"]
        ).sum(),
        "stored transaction count mismatch": customer_transaction_check[
            "count_mismatch"
        ].sum(),
    },
    name="flagged_records",
)

transaction_checks = pd.concat(
    [
        pd.Series(
            {
                "missing transaction_id": transactions["transaction_id"].isna().sum(),
                "missing customer_id": transactions["customer_id"].isna().sum(),
                "missing product_id": transactions["product_id"].isna().sum(),
                "duplicate transaction_id": transactions["transaction_id"].duplicated().sum(),
                "transactions with unknown customer_id": product_id_checks[
                    "transactions with unknown customer_id"
                ],
                "transactions with unknown product_id": product_id_checks[
                    "transactions with unknown product_id"
                ],
                "transaction product name mismatch": product_id_checks[
                    "transaction product_name mismatch"
                ],
                "transaction product category mismatch": product_id_checks[
                    "transaction product_category mismatch"
                ],
                "transaction date outside expected range": customer_transaction_checks[
                    "transaction date outside expected range"
                ],
                "transaction before first purchase": customer_transaction_checks[
                    "transaction before first purchase"
                ],
            }
        ),
        transaction_economics_checks,
    ],
    axis=0,
).rename("flagged_records")

product_checks = pd.concat(
    [
        pd.Series(
            {
                "missing product_id": products["product_id"].isna().sum(),
                "missing product_name": products["product_name"].isna().sum(),
                "missing product_category": products["product_category"].isna().sum(),
                "duplicate product_id": products["product_id"].duplicated().sum(),
                "duplicate product_name": products["product_name"].duplicated().sum(),
                "products without transactions": product_id_checks[
                    "products without transactions"
                ],
                "products without product events": product_id_checks[
                    "products without product events"
                ],
            }
        ),
        product_economics_checks[
            [label for label in product_economics_checks.index if label.startswith("product")]
        ],
        aggregate_consistency_checks,
    ],
    axis=0,
).rename("flagged_records")

product_event_checks = pd.concat(
    [
        pd.Series(
            {
                "missing product_id": product_events["product_id"].isna().sum(),
                "missing event_date": product_events["event_date"].isna().sum(),
                "duplicate product_id/event_date": product_event_key_duplicates,
                "product events with unknown product_id": product_id_checks[
                    "product events with unknown product_id"
                ],
                "product event product name mismatch": product_id_checks[
                    "product event product_name mismatch"
                ],
                "product event product category mismatch": product_id_checks[
                    "product event product_category mismatch"
                ],
                "product event date outside expected range": (
                    (product_events["event_date"] < EXPECTED_START_DATE)
                    | (product_events["event_date"] > EXPECTED_END_DATE)
                ).sum(),
                "missing product-day rows": max(expected_event_rows - len(product_events), 0),
                "extra product-day rows": max(len(product_events) - expected_event_rows, 0),
            }
        ),
        product_economics_checks[
            [label for label in product_economics_checks.index if "event" in label]
        ],
        funnel_validation_checks,
        inventory_checks,
    ],
    axis=0,
).rename("flagged_records")

validation_groups = {
    "customers.csv": customer_checks,
    "transactions.csv": transaction_checks,
    "products.csv": product_checks,
    "product_events.csv": product_event_checks,
}

dataset_status = {}
for dataset_name, checks in validation_groups.items():
    failed_checks = checks[checks > 0]
    dataset_status[dataset_name] = {
        "passed": failed_checks.empty,
        "failed_checks": failed_checks,
    }
    if not failed_checks.empty:
        issues.append(
            f"{dataset_name}: "
            + ", ".join(f"{name}={int(value)}" for name, value in failed_checks.items())
        )

if not (10_000 <= len(customers) <= 15_000):
    suspicious_patterns.append("Customer count is outside the expected 10,000-15,000 range.")
if transactions["transaction_date"].min() < EXPECTED_START_DATE or transactions[
    "transaction_date"
].max() > EXPECTED_END_DATE:
    suspicious_patterns.append("Transaction dates fall outside the expected two-year window.")
if product_events["event_date"].min() < EXPECTED_START_DATE or product_events[
    "event_date"
].max() > EXPECTED_END_DATE:
    suspicious_patterns.append("Product event dates fall outside the expected two-year window.")
if len(product_events) != expected_event_rows:
    suspicious_patterns.append(
        f"Product events do not contain exactly one product-day row per product; expected {expected_event_rows:,}."
    )
if not (25 <= average_order_value <= 120):
    suspicious_patterns.append("Average order value is outside a plausible apparel ecommerce range.")
if not (0.02 <= churn_rate <= 0.30):
    suspicious_patterns.append("Churn rate may be too low or too high for this synthetic setup.")
if not (0.05 <= price_change_frequency <= 0.35):
    suspicious_patterns.append("Price-change frequency may be too sparse or too common.")
if not (1.0 <= transactions_per_customer <= 8.0):
    suspicious_patterns.append("Transactions per customer may be outside the intended repeat-purchase range.")
if not (0.25 <= gross_margin_rate <= 0.75):
    suspicious_patterns.append("Gross margin rate may be outside a useful synthetic apparel range.")
if not (0.03 <= add_to_cart_rate <= 0.30):
    suspicious_patterns.append("Add-to-cart rate may be too low or too high for this synthetic funnel.")
if not (0.20 <= checkout_start_rate <= 0.80):
    suspicious_patterns.append("Checkout-start rate may be too low or too high for this synthetic funnel.")
if not (0.05 <= purchase_after_checkout_rate <= 0.60):
    suspicious_patterns.append("Purchase-after-checkout rate may be too low or too high for this synthetic funnel.")
if not (0.005 <= stockout_product_day_rate <= 0.20):
    suspicious_patterns.append("Stockout product-day rate may be too sparse or too common for inventory analysis.")
if (price_variation_support["unique_selling_prices"] > 1).sum() < len(products) * 0.80:
    suspicious_patterns.append("Too few products have selling-price variation for price-elasticity exploration.")

assumption_notes.extend(
    [
        "Product events are daily product-level aggregates, not customer-level clickstream sessions.",
        "The stockout flag represents constrained product availability, such as limited sizes or colors, rather than only zero on-hand inventory.",
        "Gross margin uses synthetic unit cost and excludes fulfillment, returns, marketing costs, and taxes.",
    ]
)

print("Validation Findings")
print("===================")
print(
    "The expanded synthetic ecommerce dataset contains "
    f"{len(customers):,} customers, {len(transactions):,} transactions, "
    f"{len(products):,} products, and {len(product_events):,} product-day event rows."
)
print(
    "Transactions run from "
    f"{transactions['transaction_date'].min().date()} to "
    f"{transactions['transaction_date'].max().date()}; product events run from "
    f"{product_events['event_date'].min().date()} to "
    f"{product_events['event_date'].max().date()}."
)
print(
    "Key business metrics look coherent for this synthetic setup: "
    f"average order value is ${average_order_value:,.2f}, gross margin rate is {gross_margin_rate:.1%}, "
    f"customer churn is {churn_rate:.1%}, and transaction-level price-change exposure is {price_change_frequency:.1%}."
)

print("\nStructural validation by dataset:")
for dataset_name, result in dataset_status.items():
    if result["passed"]:
        print(f"- {dataset_name}: PASSED structural validation.")
    else:
        print(f"- {dataset_name}: FAILED structural validation.")
        for check_name, value in result["failed_checks"].items():
            print(f"  - {check_name}: {int(value):,} flagged records")

print("\nFunnel-rate denominator check:")
print(f"- Add-to-cart rate: {add_to_cart_rate:.1%} = add_to_cart_events / product_views")
print(f"- Checkout-start rate: {checkout_start_rate:.1%} = checkout_started_events / add_to_cart_events")
print(f"- Purchase-after-checkout rate: {purchase_after_checkout_rate:.1%} = purchases / checkout_started_events")
print(f"- Product-day stockout rate: {stockout_product_day_rate:.1%}")

print("\nWarnings and assumptions:")
if suspicious_patterns:
    print("Suspicious values to review:")
    for pattern in suspicious_patterns:
        print(f"- {pattern}")
else:
    print("No suspicious values triggered the validation thresholds.")
print("Questionable assumptions to keep visible in later analysis:")
for note in assumption_notes:
    print(f"- {note}")

print("\nOverall conclusion:")
if issues:
    print("One or more datasets failed structural validation. Resolve the flagged records before downstream analysis.")
elif suspicious_patterns:
    print("All datasets passed structural validation, but the warning list should be reviewed before downstream analysis.")
else:
    print("All datasets passed structural validation.")
    print("The expanded dataset is internally consistent and ready for downstream analysis.")


Validation Findings
The expanded synthetic ecommerce dataset contains 12,500 customers, 48,769 transactions, 41 products, and 29,971 product-day event rows.
Transactions run from 2024-01-01 to 2025-12-31; product events run from 2024-01-01 to 2025-12-31.
Key business metrics look coherent for this synthetic setup: average order value is $57.53, gross margin rate is 53.3%, customer churn is 7.3%, and transaction-level price-change exposure is 8.8%.

Structural validation by dataset:
- customers.csv: PASSED structural validation.
- transactions.csv: PASSED structural validation.
- products.csv: PASSED structural validation.
- product_events.csv: PASSED structural validation.

Funnel-rate denominator check:
- Add-to-cart rate: 10.0% = add_to_cart_events / product_views
- Checkout-start rate: 40.0% = checkout_started_events / add_to_cart_events
- Purchase-after-checkout rate: 18.0% = purchases / checkout_started_events
- Product-day stockout rate: 5.6%

Warnings and assumptions:
No suspici